# Marketing Campaign Analysis Notebook (EN)

This notebook covers:
1. Data loading and quality checks.
2. KPI computation.
3. Segmented analysis by campaign and channel.
4. Visual analysis and executive conclusions.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

base_dir = Path.cwd()
candidates = [
    base_dir / 'Marketing_Campaign_Data.csv',
    base_dir / 'Marketing Analysis' / 'Marketing_Campaign_Data.csv',
    base_dir.parent / 'Marketing_Campaign_Data.csv',
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find Marketing_Campaign_Data.csv')

df = pd.read_csv(data_path)
df = df.rename(columns={
    'Interaction ID': 'interaction_id',
    'Campaign Type': 'campaign',
    'Channel': 'channel',
    'Customer Type': 'customer_type',
    'Converted (1=yes, 0=no)': 'converted',
    'Time on Site (seconds)': 'time_on_site',
    'Sales ($)': 'sales'
})

sns.set_theme(style='whitegrid')
df.head()

In [ ]:
kpis = {
    'interactions': len(df),
    'total_revenue': df['sales'].sum(),
    'conversion_rate': df['converted'].mean(),
    'avg_time': df['time_on_site'].mean(),
    'avg_ticket': df.loc[df['converted'] == 1, 'sales'].mean(),
}
kpis

In [ ]:
by_channel = df.groupby('channel', as_index=False).agg(
    interactions=('interaction_id', 'count'),
    conversion_rate=('converted', 'mean'),
    revenue=('sales', 'sum')
).sort_values('revenue', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=by_channel, x='channel', y='revenue')
plt.title('Revenue by Channel')
plt.show()

In [ ]:
by_combo = df.groupby(['campaign','channel'], as_index=False).agg(
    interactions=('interaction_id','count'),
    conversion_rate=('converted','mean'),
    revenue=('sales','sum')
).sort_values('revenue', ascending=False)

top = by_combo.iloc[0]
print('Top combination:', top['campaign'], '+', top['channel'])
print('Revenue:', round(top['revenue'], 2))